# Lab 2: Language Model Systems

## At a glance

One eval run says step-by-step prompting made a small model better at math. Someone reading only that number would
ship it. In this lab you test that judgment on GSM8K's grade-school word problems: is the gain real, what produced it,
and what does it cost? Fine-tuning and quantization are two more answers to the same question, one bought with
training and one paid for in accuracy.

You write five short functions, run the model with them, and predict each result before you see it. Every graded
value comes from your functions on inputs the course page issues, or from results saved in advance, so a session
without a GPU can still finish every checkpoint.
Without a GPU, tasks 1 and 2 rescore and compare real saved answers; tasks 3 to 5 run on small tensors, followed by
saved model results. With a GPU, tasks 3 to 5 also participate in the live training and quantization runs.

| | |
|---|---|
| Runtime | pilot estimate: about 35 minutes of T4 GPU compute, plus time for coding and interpretation |
| Model | Qwen2.5-0.5B-Instruct, a 0.5-billion-parameter chat model |
| Data | 1,319 GSM8K test problems: 50 for checking runs; the remaining 1,269 include the 200-problem prompting subset |

> Save your own copy first: File → Save a copy in Drive. Then choose Runtime → Change runtime type → T4 GPU.

> ### <font color="#A31F34">Big picture</font>
>
> An accuracy is a count of right answers out of n problems, read by a grader. Before it means anything, you need the
> grader to read answers correctly, the comparison to be made on the same problems, and an interval for uncertainty
> beyond those problems. This lab builds each of those, then uses them to judge three ways of changing a model.

## What is graded

- Checkpoint 1, readiness (20%): C1 to C5 from the Changed cases cell, and D1 and D2 from the Report values cell.
- Checkpoint 2, experiment record (30%): what a record of the prompting runs has to hold for someone else to check
  that S1 beats S0.
- Checkpoint 3, claim, evidence and caveat (35%): which statements a second comparison supports, run with two output
  limits that checkpoint 3 gives you.
- Checkpoint 4, next experiment (15%): what has to stay fixed when the comparison is repeated on another math dataset.

The numbers from your own GPU runs depend on your session, so they are not graded. They are there to test your
predictions.

> ### <font color="#B45309">Watch out</font>
>
> The 50 dev problems are for checking that your run works. Use the supplied settings for the lab's comparisons.
> In a new experiment, choose settings on development data, then evaluate on a separate test set. Choosing settings
> by their test scores makes the winning test score an optimistic estimate.

In [ ]:
#@title Download the lab's data and code (run this cell; there is no need to read it)
import hashlib, importlib.util, os, shutil, subprocess, sys, urllib.request

BASE = "https://raw.githubusercontent.com/codey-m/cv_nlp/main/labs/data/lab2/"
DIGESTS = {'lab2_helpers.py': '8bc7e41322a1a387f11e4159b59d754543d8862eb2e5c8b18cd4c11345074020', 'lab2_data.json.gz': '5f6918522946294a1d7a0efa80636277f90c22f630dcb64ace60a7af24ef2f8c', 'reference_adapter/adapter_config.json': '29440610008c99637c73fc36f25c452bae97f1215d8e73d338eac39e23ef84c9', 'reference_adapter/adapter_model.safetensors': 'faaef095ebac9ecc9f8ca0fdc09f16259df2470184979247b37cd7f0f190dd9c'}

for name, digest in DIGESTS.items():
    ok = os.path.exists(name) and hashlib.sha256(open(name, "rb").read()).hexdigest() == digest
    if not ok:
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        source = os.environ.get("LAB2_BASE", BASE)
        if source.startswith("http"):
            urllib.request.urlretrieve(source + name, name)
        else:
            shutil.copyfile(os.path.join(source, name), name)
    if hashlib.sha256(open(name, "rb").read()).hexdigest() != digest:
        raise RuntimeError(f"{name} did not download intact. Delete it and run this cell again.")
import lab2_helpers as H
DATA = H.load_data("lab2_data.json.gz")
GPU = H.torch.cuda.is_available()
if GPU and importlib.util.find_spec("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
# peft refuses to build LoRA layers beside an older torchao, which Colab has shipped. This lab never uses torchao.
if GPU and subprocess.run([sys.executable, "-c", "from peft.import_utils import is_torchao_available as f; f()"],
                          capture_output=True).returncode:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=True)
print(f"{len(DATA['full'])} test problems, {len(DATA['lora_train'])} training solutions;",
      "GPU:", H.torch.cuda.get_device_name(0) if GPU else "none, so the GPU cells will load saved results")

In [ ]:
import math
import re
import torch

## Setup: the problems and the three systems

Every system is the same model with plain greedy decoding, at most 512 output tokens and no repetition penalty. The
model's own defaults sample and penalize repeats, so every run here sets decoding explicitly. The systems differ only
in the chat they are given.

| System | Its chat |
|---|---|
| S0, the control | a system prompt asking for a single number, then the problem |
| S1, step by step | a system prompt asking for step-by-step working that ends "The answer is N.", then the problem |
| S2, worked examples | S1's chat with four solved problems as earlier turns |

Read five of S0's answers on the dev problems, then one of S1's.

In [ ]:
dev = DATA["runs_dev"]["S0"]
for k in range(5):
    p = DATA["problems"][DATA["dev"][k]]
    print(f"S0 answered {dev['raw'][k].strip()!r:12s} right answer {p['answer']}")
print()
print(H.SYSTEMS["S1"]["system"])
print(DATA["runs"]["S1"]["raw"][0])

## Your task 1: Read the final answer

The grader needs one number from each answer. Write `extract_answer`: the number after the last "answer is", or else
the last number in the text, with commas, `$` and a trailing `.0` or `.00` removed. Return `None` when there is no
number. Here, "unreadable" means the extractor found no number, not that the reasoning is unfinished or wrong.
Unreadable answers count against accuracy and also appear in a column of their own.

<details><summary>Hint</summary>

A regular expression such as `-?\$?\d[\d,]*(?:\.\d+)?` finds numbers with commas, a dollar sign or a decimal
point. `str.lower().rfind("answer is")` finds the last "answer is". Return an `int` when no decimal part remains and a
`float` otherwise.

</details>

In [ ]:
def extract_answer(text):
    """The number after the last "answer is", or else the last number in the text; None if there is none."""
    # TODO: find the last "answer is" and the first number after it; failing that, the last number anywhere.
    raise NotImplementedError("Read the answer.")

In [ ]:
assert extract_answer("so she pays $1,200.00. The answer is 1200") == 1200
assert extract_answer("3 + 4 = 7, then 7 * 2 = 14") == 14
assert extract_answer("The answer is 18. Check: 18 + 2 = 20") == 18
assert extract_answer("I am not sure.") is None
assert extract_answer("The answer is -3.") == -3 and extract_answer("She has 2.5 cups") == 2.5
print("Task 1 checks pass.")

## Your task 2: Compare two systems on the same problems

`paired_comparison` takes two lists of right-or-wrong flags, one per problem, in the same order. It returns the
accuracy difference B minus A, an approximate 95% interval on it, and an exact sign test. The paired difference
depends on the two directions of disagreement and the total number of problems, not two independent accuracy estimates.
The observed scores on these fixed problems are known. The interval estimates uncertainty for comparable problems
under repeated sampling; it is not a 95% probability that B beats A on this fixed set. The normal approximation is
unreliable with very few disagreements; the exact sign test does not use that approximation.

<details><summary>Hint</summary>

Count `a_only` (A right, B wrong) and `b_only` (B right, A wrong). With n problems, the difference is
(b_only − a_only)/n and its standard error is √(a_only + b_only − (b_only − a_only)²/n)/n. The interval is the
difference ± 1.96 standard errors. `H.sign_test(a_only, b_only)` gives the exact p-value.

</details>

In [ ]:
def paired_comparison(correct_a, correct_b):
    """B minus A on the same problems: a dict with n, acc_a, acc_b, a_only, b_only, diff, lo, hi and p."""
    # TODO: count the two kinds of disagreement, then compute the difference, its 95% interval and the sign test.
    raise NotImplementedError("Compare the two runs.")

In [ ]:
c = paired_comparison([1, 1, 1, 0, 0, 0, 1, 1, 0, 1], [1, 0, 1, 1, 1, 1, 1, 1, 1, 1])
assert (c["a_only"], c["b_only"]) == (1, 4) and math.isclose(c["diff"], 0.3)
assert math.isclose(c["hi"] - c["diff"], 1.96 * math.sqrt(5 - 9 / 10) / 10)
assert math.isclose(c["p"], 2 * 6 / 32)
assert paired_comparison([1, 0], [1, 0])["lo"] == 0, "With no observed disagreements, this interval formula returns zero."
print("Task 2 checks pass.")

## Your task 3: Add up the log-probabilities that count

Fine-tuning trains on the solution tokens only, and a log-likelihood score reads them only. `masked_logprob` takes a
tensor of log-probabilities and a mask of the same shape, 1 where a token counts and 0 elsewhere, and returns the sum
and the count of the included log-probabilities along the last dimension. Return both, not their ratio: a corpus
figure is total over total, and a mean of per-example means weights short examples too heavily.

<details><summary>Hint</summary>

Multiply by the mask and sum over the last dimension; the count is the mask's sum. Convert the mask to the
log-probabilities' dtype first.

</details>

In [ ]:
def masked_logprob(log_probs, include):
    """Sum and count of log_probs where include is 1, along the last dimension."""
    # TODO: return (sum of the included log-probabilities, how many there are).
    raise NotImplementedError("Mask the log-probabilities.")

In [ ]:
s, n = masked_logprob(torch.tensor([[-100.0, -1.0, -2.0, -999.0]]), torch.tensor([[0, 1, 1, 0]]))
assert s.item() == -3 and n.item() == 2
s, n = masked_logprob(torch.tensor([[-1.0, -2.0, 0.0], [-0.5, -1.0, -1.5]]), torch.tensor([[1, 1, 0], [1, 1, 1]]))
assert math.isclose(-s.sum().item() / n.sum().item(), 1.2), "Total over total gives 1.2; a mean of means gives 1.25."
print("Task 3 checks pass.")

## Your task 4: Quantize a weight matrix to int4

`quantize_matrix` rounds each row of a weight matrix in groups of `group_size` weights. Each group gets one
scale, its largest absolute weight divided by 7, rounded to fp16 because that is how it is stored. Each weight becomes
an integer from −8 to 7 times the scale. Return the rounded weights as floats, in the input's shape and dtype, so the
model can use them directly. `group_size="row"` means one group per row.

<details><summary>Hint</summary>

Reshape to (rows, groups, group_size) with `W.float().reshape(...)`, take `.abs().amax(dim=-1, keepdim=True) / 7`,
round it with `.half().float()`, and replace a zero scale with 1. Then `torch.clamp(torch.round(W / scale), -8, 7) *
scale`. Raise `ValueError` if the group size does not divide a row.

</details>

In [ ]:
def quantize_matrix(W, group_size):
    """W rounded to int4 in groups along each row with absmax scales, returned as floats in W's shape and dtype."""
    # TODO: reshape into groups, compute each group's scale, round and clamp, then scale back and reshape.
    raise NotImplementedError("Quantize the matrix.")

In [ ]:
gen = torch.Generator().manual_seed(0)
W = torch.randn(8, 128, generator=gen)
Wq = quantize_matrix(W, 64)
assert Wq.shape == W.shape and Wq.dtype == W.dtype
assert torch.allclose(quantize_matrix(Wq, 64), Wq, atol=1e-6), "Quantizing twice should change nothing."
spiked = W.clone()
spiked[0, 0] = 50.0
err = lambda A, gs: (quantize_matrix(A, gs) - A)[0, 1:64].abs().mean().item()
assert err(spiked, 64) > 5 * err(W, 64), "One outlier should cost its whole group its precision."
assert (quantize_matrix(spiked, 64) - spiked)[0, 64:].abs().mean() < 0.2, "The next group should be untouched."
print("Task 4 checks pass.")

## Your task 5: Choose each group's scale by what it does to the output

Task 4's scale puts the largest absolute weight near seven steps. Clipping it a little rounds every
other weight more finely, and whether that helps depends on what the layer's inputs do with each weight. A layer
computes Wx, so a rounding error e = Wq − W in one row changes the output by eᵀx, and its mean square over the inputs
is eᵀΣe with Σ = E[xxᵀ]. Here we optimize each group separately: x and e contain that group's coordinates,
and Σ is its input second-moment block. This objective omits interactions between errors in different groups.
`sigma` has shape (groups, group_size, group_size); a coordinate with zero input energy contributes no output error.
`best_scales` tries fractions of task 4's scale for every group and keeps the one whose eᵀΣe is smallest.
`H.quantize_with_scales(W, scale)` rounds with given scales, and
`H.output_error(W, Wq, sigma, group_size)` returns eᵀΣe for every row and group.

<details><summary>Hint</summary>

For each factor f, the candidate scale is f × absmax / 7, rounded to fp16, with zeros replaced by 1, of shape
(rows, groups, 1). Keep a running best: `torch.where(err < best_err, ...)` updates the groups where a factor does
better. `H.group_view(W, group_size)` reshapes W to (rows, groups, group_size).

</details>

In [ ]:
def best_scales(W, sigma, group_size, factors=H.SCALE_FACTORS):
    """For every row and group, the scale among factor * absmax / 7 whose output error e^T Sigma e is smallest."""
    # TODO: try every factor, measure each group's output error, and keep the best scale for each group.
    raise NotImplementedError("Choose the scales.")

In [ ]:
W = torch.randn(16, 128, generator=gen)
A = torch.randn(2, 64, 64, generator=gen)
sigma = A @ A.transpose(1, 2) / 64 + 0.01 * torch.eye(64)
plain = best_scales(W, sigma, 64, factors=(1.0,))
assert plain.shape == (16, 2, 1)
assert torch.equal(H.quantize_with_scales(W, plain), quantize_matrix(W, 64)), "Factor 1.0 should give task 4's rounding."
chosen = best_scales(W, sigma, 64)
e_plain = H.output_error(W, H.quantize_with_scales(W, plain), sigma, 64)
e_chosen = H.output_error(W, H.quantize_with_scales(W, chosen), sigma, 64)
assert (e_chosen <= e_plain + 1e-6).all() and e_chosen.sum() < e_plain.sum()
spiked = W.clone()
spiked[:, 0] = 8.0
unused = torch.zeros(2, 64, 64)
unused[:, 1:, 1:] = torch.eye(63)
assert (best_scales(spiked, unused, 64)[:, 0] < best_scales(spiked, unused, 64, factors=(1.0,))[:, 0]).all(), \
    "When no input reaches the outlier, clipping it should win."
print("Task 5 checks pass.")

## Changed cases

Before trusting your functions on the model, run them on inputs they have not seen. Replace the values at the top of
the next cell with the ones on the course page for checkpoint 1, run it, and enter C1 to C5 as it prints them.

In [ ]:
# Replace these values with the inputs from the course page, then run the cell.
C1_ANSWER = "It costs $3 a day, so 7 days cost $21. The answer is 21."
C2_CORRECT_A = [1, 0, 1, 1, 0, 1, 0, 1]
C2_CORRECT_B = [1, 1, 1, 0, 1, 1, 1, 1]
C3_LOG_PROBS = [[-0.5, -1.0, -2.0], [-0.25, -0.75, -4.0]]
C3_INCLUDE = [[0, 1, 1], [1, 1, 0]]
C4_ROW = [0.5, -1.0, 2.0, 0.25]
C4_GROUP_SIZE = 2
C5_ROW = [7.0, 1.2, -0.4, 0.3]
C5_SIGMA_DIAGONAL = [0.0, 1.0, 1.0, 1.0]

print("C1 =", extract_answer(C1_ANSWER))
print("C2 =", round(paired_comparison(C2_CORRECT_A, C2_CORRECT_B)["hi"], 6))
s3, n3 = masked_logprob(torch.tensor(C3_LOG_PROBS), torch.tensor(C3_INCLUDE))
print("C3 =", round(-s3.sum().item() / n3.sum().item(), 6))
W4 = torch.tensor([C4_ROW])
print("C4 =", round((quantize_matrix(W4, C4_GROUP_SIZE) - W4).abs().sum().item(), 6))
W5 = torch.tensor([C5_ROW])
sig5 = torch.diag(torch.tensor(C5_SIGMA_DIAGONAL))[None]
print("C5 =", round((best_scales(W5, sig5, len(C5_ROW)) / best_scales(W5, sig5, len(C5_ROW), factors=(1.0,))).item(), 6))

## Provided: the prompting runs

Each run below is the model answering 200 test problems. On a GPU it runs live, in about five minutes for all three;
without one the cell loads the same runs from a T4. The record reads every answer with your `extract_answer`.
The table's accuracy is a fraction; prompt and output are mean tokens per problem. GFLOPs is projected compute
at 2N operations per token, where N is the model's parameter count, not measured latency.

**Predict first:** S0 answers in a single number. What accuracy will S1, which writes out its steps, reach, and will
four worked examples (S2) add to it? Enter your guesses as fractions.

In [ ]:
# STUDENT PREDICTIONS: replace None with your guesses, as fractions between 0 and 1.
PREDICT_S1 = None
PREDICT_S2 = None

In [ ]:
assert all(isinstance(x, (int, float)) and 0 <= x <= 1 for x in (PREDICT_S1, PREDICT_S2)), "Enter fractions from 0 to 1."
TEST200 = DATA["test200"]
if GPU:
    model, tok = H.load_model()
    p200 = {name: H.run_system(name, model, tok, DATA, TEST200, H.SYSTEMS[name], extract_answer) for name in H.SYSTEMS}
else:
    p200 = {name: H.cached(DATA, name, extract_answer, "200") for name in H.SYSTEMS}

print("\nEvidence Record: Qwen2.5-0.5B-Instruct, greedy, 512 output tokens, on the same 200 test problems")
H.show(p200.values())
print()
H.show_comparison("S1 vs S0", paired_comparison(p200["S0"]["correct"], p200["S1"]["correct"]))
H.show_comparison("S2 vs S1", paired_comparison(p200["S1"]["correct"], p200["S2"]["correct"]))
acc = {name: H.summary(run)["accuracy"] for name, run in p200.items()}
print(f"\nyou predicted S1 {PREDICT_S1:.2f} and S2 {PREDICT_S2:.2f}; the record shows S1 {acc['S1']:.3f} and S2 {acc['S2']:.3f}")

## Provided: the reference runs on 1,269 problems

The rest of the lab compares on the 1,269 test problems outside dev, where an interval is about 2.5 times narrower
than on 200. These three runs were made on a T4 in advance, with the same settings as yours.

In [ ]:
ref = {name: H.cached(DATA, name, extract_answer) for name in H.SYSTEMS}
H.show(ref.values())
print()
H.show_comparison("S1 vs S0, 1,269 problems", paired_comparison(ref["S0"]["correct"], ref["S1"]["correct"]))
H.show_comparison("S2 vs S1, 1,269 problems", paired_comparison(ref["S1"]["correct"], ref["S2"]["correct"]))

## Provided: the rerun

Greedy decoding gives one answer per problem. Sampling, at temperature 0.7 and top-p 0.9 as in lecture 2.1, gives a
potentially different answer each time. Two sampled runs of S1 on the 1,269 problems were saved.

**Predict first:** on how many of the 1,269 problems will the two runs disagree, one right and the other wrong?

In [ ]:
PREDICT_DISAGREE = None   # a whole number of problems

In [ ]:
assert isinstance(PREDICT_DISAGREE, int), "Enter a whole number."
rerun = [H.cached(DATA, name, extract_answer) for name in ("S1~1", "S1~2")]
H.show(rerun)
r = paired_comparison(rerun[0]["correct"], rerun[1]["correct"])
H.show_comparison("second sampled run vs first", r)
print(f"\nthey disagree on {r['a_only'] + r['b_only']} problems; you predicted {PREDICT_DISAGREE}")

## Report values

Before revealing D1, predict the weight-memory ratio. A packed int4 format uses 4 bits per quantized weight,
plus one 16-bit scale per group of 64. What fraction of the model's fp16 weight memory do you expect it to need?

Then run the report and enter D1 and D2 in checkpoint 1. They come from the fixed model configuration and the saved
reruns, so this is part of the no-GPU path. D1 is a storage projection, not a measurement of this notebook's GPU memory.

In [ ]:
PREDICT_RATIO = None   # a fraction between 0 and 1

In [ ]:
assert isinstance(PREDICT_RATIO, (int, float)) and 0 < PREDICT_RATIO < 1, "Enter a fraction between 0 and 1."

In [ ]:
REPORT_WEIGHTS = H.cached_weight_bytes(64)
H.show_weight_bytes(REPORT_WEIGHTS, PREDICT_RATIO)
D1 = REPORT_WEIGHTS["ratio"]
D2 = r["a_only"] + r["b_only"]
print("D1 (projected int4 weight bytes over fp16 weight bytes, group 64) =", round(D1, 6))
print("D2 (problems on which the two sampled reruns of S1 disagree) =", D2)

Compare your prediction with the per-part memory table. Which part explains the gap?

## A second comparison, for checkpoint 3

S0 and S1 differ only in their instruction, and S1 spends about six times as many output tokens. Checkpoint 3 gives
you two output limits for S1. Replace the values at the top of the next cell with them and run it. It scores S1 at
each limit against S0 and against S1 at 512, from saved runs, so it needs no GPU.

**Predict first:** if S1 may write only half as much, how much of its gain over S0 survives? Write a fraction and
one reason in a text cell before running the comparison; then compare your prediction with the reported share.

In [ ]:
# Replace these with the two output limits from the course page for checkpoint 3, then run the cell.
CHECK_LIMITS = [64, 512]

assert all(l in (64, 128, 192, 256, 384, 512) for l in CHECK_LIMITS), "Use limits from 64, 128, 192, 256, 384 and 512."
limited = [ref["S1"] if l == 512 else H.cached(DATA, f"S1@{l}", extract_answer) for l in CHECK_LIMITS]
for run, l in zip(limited, CHECK_LIMITS):
    run["name"] = f"S1 at {l} tokens"
H.show([ref["S0"], *limited])
print()
gain = H.summary(ref["S1"])["accuracy"] - H.summary(ref["S0"])["accuracy"]
for run in limited:
    H.show_comparison(f"{run['name']} vs S0", paired_comparison(ref["S0"]["correct"], run["correct"]))
    H.show_comparison(f"{run['name']} vs S1", paired_comparison(ref["S1"]["correct"], run["correct"]))
    kept = (H.summary(run)["accuracy"] - H.summary(ref["S0"])["accuracy"]) / gain
    print(f"{run['name']} keeps {kept:.0%} of S1's gain over S0; "
          f"{sum(not f for f in run['finished'])} of its answers hit the limit\n")

## Read the evidence (ungraded)

**Audit the grader's errors and S1's errors.** Start with three contrasting real outputs. Read each one, then commit
to a label and quote the first line that supports your diagnosis. Only after all three commitments will the notebook
show its review. If the model's answer is right, quote where the extractor misreads it instead. Showing S0's answer
beside S1's is deliberate.

| Label | What it means |
|---|---|
| Arithmetic slip | the needed calculation is set up correctly, but evaluated incorrectly |
| Wrong setup | the calculation or interpretation is wrong, even if its arithmetic is correct |
| Incomplete generation | the response repeats or stops before completing a stable answer |
| Extraction failure | the model's answer is right, but the grader reads a different number |

These cases were selected to contrast failure mechanisms. Their counts do not estimate how common each mechanism is.

In [ ]:
GUIDED_AUDIT = [1, 2, 3]
for audit_item in GUIDED_AUDIT:
    H.show_audit(DATA, extract_answer, audit_item)
    print("\n" + "=" * 88 + "\n")

In [ ]:
# Before running the reveal, enter a label and quote the first line that supports it for all three cases.
MY_AUDIT = {
    1: {"label": None, "evidence": ""},
    2: {"label": None, "evidence": ""},
    3: {"label": None, "evidence": ""},
}

In [ ]:
assert set(MY_AUDIT) == set(GUIDED_AUDIT)
assert all(row["label"] in H.AUDIT_LABELS and row["evidence"].strip() for row in MY_AUDIT.values()), \
    "Give every guided case one of the four labels and quote its first supporting line."
for audit_item in GUIDED_AUDIT:
    mine = MY_AUDIT[audit_item]
    print(f"Your item {audit_item} diagnosis: {mine['label']}: {mine['evidence']}")
    H.show_audit_feedback(DATA, audit_item)
    print()

In [ ]:
# After the reveal, name the case that most changed your diagnosis and say what evidence changed it.
AUDIT_REVISION = ""

In [ ]:
assert len(AUDIT_REVISION.strip()) >= 20, "Name a case and the evidence that changed or confirmed your diagnosis."
print(AUDIT_REVISION)

**Explore one more case.** Choose an item from 4 to 10. Record a diagnosis before running the feedback cell. Item 9
is the incomplete response; the others show different setup and interpretation errors.

In [ ]:
EXPLORE_ITEM = 4
assert 4 <= EXPLORE_ITEM <= 10
H.show_audit(DATA, extract_answer, EXPLORE_ITEM)

In [ ]:
EXTRA_AUDIT_LABEL = None
EXTRA_AUDIT_EVIDENCE = ""

In [ ]:
assert EXTRA_AUDIT_LABEL in H.AUDIT_LABELS and EXTRA_AUDIT_EVIDENCE.strip(), \
    "Choose a label and quote the first supporting line before revealing the review."
print(f"Your diagnosis: {EXTRA_AUDIT_LABEL}: {EXTRA_AUDIT_EVIDENCE}")
H.show_audit_feedback(DATA, EXPLORE_ITEM)

Now look back at the unreadable and output-limit columns. An extractor can return a number from an incomplete answer,
and it can misread a complete, correct one. What would happen to system accuracy in each case? Which additional
column or audit would keep those two failures visible?

## Provided: can the answer reader change a system ranking?

The extraction failure above is one case. Now test its system-level consequence on the same 1,269 problems.
`S2-self` uses S1's prompt plus four worked examples generated by this model. Its responses often put an answer in
`\boxed{...}`, so a reader that checks the last box before falling back to the strict rule is also defensible.

This is a sensitivity analysis, not a replacement for your graded `extract_answer`: the problems, model outputs and
gold answers stay fixed, and both systems are rescored by the same alternate rule. Predict the change before the
reveal. Will box-first reading improve S2-self, and will it move from below S1 to above it?

In [ ]:
BOXED = re.compile(r"\\boxed\{([^{}]+)\}")

def extract_boxed_then_strict(text):
    """Read the last simple boxed answer, then fall back to the lab's strict extractor."""
    for content in reversed(BOXED.findall(text)):
        answer = extract_answer(content)
        if answer is not None:
            return answer
    return extract_answer(text)

In [ ]:
# STUDENT PREDICTIONS: a fraction for the S2-self accuracy gain, then True or False for the ordering.
PREDICT_BOXED_GAIN = None
PREDICT_ORDER_FLIPS = None

In [ ]:
assert isinstance(PREDICT_BOXED_GAIN, (int, float)) and -1 <= PREDICT_BOXED_GAIN <= 1, \
    "Enter the predicted accuracy change as a fraction between -1 and 1."
assert isinstance(PREDICT_ORDER_FLIPS, bool), "Enter True or False for the ordering prediction."

s2_self_strict = H.cached(DATA, "S2-self", extract_answer)
s2_self_strict["name"] = "S2 self, strict"
s1_boxed = H.cached(DATA, "S1", extract_boxed_then_strict)
s1_boxed["name"] = "S1, box first"
s2_self_boxed = H.cached(DATA, "S2-self", extract_boxed_then_strict)
s2_self_boxed["name"] = "S2 self, box first"

H.show([ref["S1"], s2_self_strict, s1_boxed, s2_self_boxed])
strict_cmp = paired_comparison(ref["S1"]["correct"], s2_self_strict["correct"])
boxed_cmp = paired_comparison(s1_boxed["correct"], s2_self_boxed["correct"])
print()
H.show_comparison("S2-self vs S1, strict", strict_cmp)
H.show_comparison("S2-self vs S1, box first", boxed_cmp)

boxed_gain = H.summary(s2_self_boxed)["accuracy"] - H.summary(s2_self_strict)["accuracy"]
ordering_flipped = strict_cmp["diff"] < 0 and boxed_cmp["diff"] > 0
recovered = sum(not a and b for a, b in zip(s2_self_strict["correct"], s2_self_boxed["correct"]))
lost = sum(a and not b for a, b in zip(s2_self_strict["correct"], s2_self_boxed["correct"]))
print(f"\nS2-self box-first gain: {boxed_gain:+.3f} ({recovered} recovered, {lost} lost); "
      f"you predicted {PREDICT_BOXED_GAIN:+.3f}.")
print(f"Point ordering flipped: {ordering_flipped}; you predicted {PREDICT_ORDER_FLIPS}. "
      "Read the paired intervals before treating either ordering as settled.")

## Provided: fine-tuning

Keep S1's prompt fixed and add training. LoRA adds a low-rank update to
every attention and MLP weight and trains only that. The training set is the model's own solutions to 330 GSM8K
training problems: S1 was sampled four times on each, and a sample that ended with the right answer was kept. The loss
counts only the solution tokens, through your `masked_logprob`.

Training costs about 4N FLOPs per token it reads, where N is the parameter count: the forward pass, plus the backward
pass through the frozen weights. S2 pays for its worked examples on every call, at 2N per extra prompt token.

**Predict first:** after training, will S1 + LoRA beat S1 on the 1,269 problems? Write your expected accuracy gain
and one reason in a text cell; then check the gain and paired interval for each saved seed.

In [ ]:
if GPU:
    model, tok = H.load_model()
    N = H.cost_facts(model)["params"]
    train = H.solution_examples(tok, DATA["lora_train"])
    TRAIN_TOKENS = sum(len(e["input_ids"]) for e in train) * H.LORA_CONFIG["epochs"]
    del model
    H.free()
else:
    N = REPORT_WEIGHTS["fp16"] // 2
    TRAIN_TOKENS = H.CACHED_TRAIN_TOKENS
print(f"{len(DATA['lora_train'])} training solutions, read {H.LORA_CONFIG['epochs']} times: {TRAIN_TOKENS:,} tokens; N = {N:,}")
print(f"projected training cost: {4 * N * TRAIN_TOKENS / 1e15:.2f} PFLOPs at 4N per token")

In [ ]:
if GPU:
    base, tok = H.load_model()
    tuned, losses = H.train_lora(base, tok, train, masked_logprob)
    if not (losses[-1] < losses[0] and all(math.isfinite(l) for l in losses)):
        print("Training did not converge in this session, so the reference adapter stands in for it.")
        base, tok = H.load_model()
        tuned = H.load_adapter(base, "reference_adapter")
    tuned = tuned.merge_and_unload()
    lora = H.run_system("S1 + LoRA", tuned, tok, DATA, DATA["full"], H.SYSTEMS["S1"], extract_answer)
    del tuned, base
    H.free()
else:
    lora = H.cached(DATA, "LoRA-self-s0", extract_answer)
    lora["name"] = "S1 + LoRA"

In [ ]:
seeds = [H.cached(DATA, f"LoRA-self-s{s}", extract_answer) for s in range(3)]
human = H.cached(DATA, "LoRA-human-s0", extract_answer)
human["name"] = "LoRA on GSM8K's solutions"
H.show([ref["S1"], ref["S2"], lora, *seeds, human])
print()
H.show_comparison("your S1 + LoRA vs S1", paired_comparison(ref["S1"]["correct"], lora["correct"]))
for s, run in enumerate(seeds):
    H.show_comparison(f"saved seed {s} vs S1", paired_comparison(ref["S1"]["correct"], run["correct"]))
H.show_comparison("GSM8K's solutions vs the model's own", paired_comparison(seeds[0]["correct"], human["correct"]))
extra_prompt = H.summary(ref["S2"])["prompt_tokens"] - H.summary(ref["S1"])["prompt_tokens"]
print(f"\nTraining versus extra prompt cost: training read {TRAIN_TOKENS:,} tokens at 4N; "
      f"S2 adds {extra_prompt:.0f} prompt tokens per call at 2N. These costs match after "
      f"{4 * TRAIN_TOKENS / (2 * extra_prompt):.0f} calls, excluding training-data generation and output-token differences.")

The strict grader gives saved LoRA seed 0 a small edge over S2-self whose paired interval excludes zero. The audit
suggests one last sensitivity question: if both runs use box-first reading, will that interval still exclude zero?
Commit to a prediction before revealing the rescored comparison.

In [ ]:
PREDICT_LORA_EDGE_SURVIVES = None

In [ ]:
assert isinstance(PREDICT_LORA_EDGE_SURVIVES, bool), "Enter True or False."
lora0_boxed = H.cached(DATA, "LoRA-self-s0", extract_boxed_then_strict)
strict_lora_cmp = paired_comparison(s2_self_strict["correct"], seeds[0]["correct"])
boxed_lora_cmp = paired_comparison(s2_self_boxed["correct"], lora0_boxed["correct"])
H.show_comparison("LoRA seed 0 vs S2-self, strict", strict_lora_cmp)
H.show_comparison("LoRA seed 0 vs S2-self, box", boxed_lora_cmp)
edge_survives = boxed_lora_cmp["lo"] > 0
print(f"Box-first interval still excludes zero on the positive side: {edge_survives}; "
      f"you predicted {PREDICT_LORA_EDGE_SURVIVES}.")

## Provided: quantization

Weight memory is one part of serving cost. Revisit your prediction and the D1 table in Report values.
The projected format packs quantized linear weights into 4 bits plus group scales and leaves other weights in fp16.
Here we simulate its rounding in floating-point tensors: the accuracy changes are measured, but storage savings
are projected. This notebook does not measure the speed or GPU-memory use of a packed-int4 implementation.

In [ ]:
if GPU:
    model, tok = H.load_model()
    WEIGHTS = H.weight_bytes(model, 64)
else:
    WEIGHTS = REPORT_WEIGHTS
H.show_weight_bytes(WEIGHTS, PREDICT_RATIO)

Now quantize the model with task 4, then again with task 5's scales, and score both. The log-likelihood of GSM8K's
written solutions needs one forward pass per problem, so it is fast and sensitive to small damage. Accuracy needs a
full generation run. On a GPU both run live; without one the cell reads the saved log-likelihood and accuracy rows.

Task 5 needs Σ for every layer. It comes from the unquantized model's inputs on the 128 calibration solutions, which
are training problems, never test ones.

**Predict first:** task 5 changes only the scales, not the 4 bits. What fraction of int4's log-likelihood loss will it
win back? Write your fraction and reason in a text cell, then compare with the recovery reported below.

In [ ]:
if GPU:
    calib = H.solution_examples(tok, DATA["calibration"])
    ll_set = H.solution_examples(tok, [{"id": i, **DATA["problems"][i]} for i in DATA["full"]])
    ll = {"fp16": H.solution_loglik(model, tok, ll_set, masked_logprob)}
    sigmas = H.input_moments(model, tok, calib, 64)

    H.quantize_model(model, quantize_matrix, 64)
    ll["int4"] = H.solution_loglik(model, tok, ll_set, masked_logprob)
    int4 = H.run_system("S1 int4", model, tok, DATA, DATA["full"], H.SYSTEMS["S1"], extract_answer)
    del model
    H.free()

    model, tok = H.load_model()
    search = H.quantize_model_calibrated(model, sigmas, 64, best_scales)
    ll["int4, calibrated"] = H.solution_loglik(model, tok, ll_set, masked_logprob)
    cal = H.run_system("S1 int4, calibrated", model, tok, DATA, DATA["full"], H.SYSTEMS["S1"], extract_answer)
    del model, sigmas
    H.free()
    ll_n = len(ll_set)
    print(f"task 5 clipped {search['clipped_share']:.0%} of groups and cut the summed output error to "
          f"{search['error_ratio']:.2f} of task 4's")
else:
    ll = {"fp16": tuple(DATA["log_likelihood"]["base"][k] for k in ("sums", "counts")),
          "int4": tuple(DATA["log_likelihood"]["int4"][k] for k in ("sums", "counts")),
          "int4, calibrated": tuple(DATA["log_likelihood"]["int4_cal"][k] for k in ("sums", "counts"))}
    int4 = {**H.cached(DATA, "S1-int4-g64", extract_answer), "name": "S1 int4"}
    cal = {**H.cached(DATA, "S1-int4-g64-cal", extract_answer), "name": "S1 int4, calibrated"}
    ll_n = len(ll["fp16"][0])
    print("No GPU: showing saved quantization results. Task 4 and task 5 already ran on their toy checks.")

In [ ]:
gb = {"fp16": WEIGHTS["fp16"] / 1e9, "S1 int4": WEIGHTS["int4"] / 1e9, "S1 int4, calibrated": WEIGHTS["int4"] / 1e9}
H.show([ref["S1"], int4, cal], weight_gb=gb)
print()
H.show_comparison("int4 vs S1", paired_comparison(ref["S1"]["correct"], int4["correct"]))
H.show_comparison("calibrated vs int4", paired_comparison(int4["correct"], cal["correct"]))
print(f"\nlog-likelihood of the gold solutions, nats per token, on {ll_n:,} problems:")
for name, (sums, counts) in ll.items():
    print(f"  {name:18s} {H.per_token(sums, counts):.4f}")
lost = H.per_token(*ll["fp16"]) - H.per_token(*ll["int4"])
back = H.per_token(*ll["int4, calibrated"]) - H.per_token(*ll["int4"])
print(f"calibration wins back {back / lost:.0%} of the log-likelihood int4 lost")

## Further reflection (ungraded)

Which result was furthest from your prediction? Return to the opening decision: would you deploy a change to S0?
Name the system and reference you would compare, cite an accuracy difference with its paired interval, and state
the cost you would accept. Name one result you still need before making that decision for another dataset.